In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create dummy dataset
data = pd.DataFrame({
    'hours': [1,2,3,4,5,6,7,8,9,10]*10,
    'attendance': [50,55,60,65,70,75,80,85,90,95]*10,
    'marks': [20,25,30,35,40,50,60,70,80,90]*10
})

X = data[['hours', 'attendance']]
y = data['marks']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [2]:
import numpy as np

NUM_CLIENTS = 5
client_data = []

X_split = np.array_split(X, NUM_CLIENTS)
y_split = np.array_split(y, NUM_CLIENTS)

for i in range(NUM_CLIENTS):
    client_data.append((X_split[i], y_split[i]))

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


In [3]:
class LinearModel:
    def __init__(self, input_dim):
        self.weights = np.zeros(input_dim)
        self.bias = 0

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

    def train(self, X, y, lr=0.01, epochs=10):
        n = len(X)
        for _ in range(epochs):
            y_pred = self.predict(X)
            
            dw = (1/n) * np.dot(X.T, (y_pred - y))
            db = (1/n) * np.sum(y_pred - y)

            self.weights -= lr * dw
            self.bias -= lr * db

In [4]:
def client_update(model, X, y):
    local_model = LinearModel(len(model.weights))
    local_model.weights = model.weights.copy()
    local_model.bias = model.bias

    local_model.train(X, y, epochs=20)
    
    return local_model.weights, local_model.bias

In [5]:
def aggregate(updates):
    avg_weights = np.mean([u[0] for u in updates], axis=0)
    avg_bias = np.mean([u[1] for u in updates])
    return avg_weights, avg_bias

In [6]:
global_model = LinearModel(input_dim=2)

ROUNDS = 10

for r in range(ROUNDS):
    updates = []

    for client_X, client_y in client_data:
        w, b = client_update(global_model, client_X, client_y)
        updates.append((w, b))

    # Server aggregation
    new_weights, new_bias = aggregate(updates)

    global_model.weights = new_weights
    global_model.bias = new_bias

    print(f"Round {r+1} completed")

Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed
Round 5 completed
Round 6 completed
Round 7 completed
Round 8 completed
Round 9 completed
Round 10 completed


In [7]:
from sklearn.metrics import mean_squared_error

preds = global_model.predict(X)
mse = mean_squared_error(y, preds)

print("Final MSE:", mse)

Final MSE: 57.91358852495011
